In [ ]:
!pip install langchain chromadb faiss-cpu openai tiktoken langchain_openai langchain-community wikipedia
!pip install rank_bm25
!pip install pypdf
!pip install langchain_community
!pip install bitsandbytes
!pip install accelerate
!pip install weaviate-client
!pip install cohere
!pip install sentence_transformers
!pip install llama-cpp-python
!pip -q install google-generativeai
!pip install unstructured
!pip install flashrank
!pip install faiss-cpu
!pip install lark
%%time
%pip install --upgrade --quiet  langchain-google-genai

In [ ]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader, CSVLoader, DirectoryLoader, WebBaseLoader
from sentence_transformers import SentenceTransformer
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain.retrievers.self_query.chroma import ChromaTranslator
from langchain.text_splitter import RecursiveCharacterTextSplitter,CharacterTextSplitter, Language, SemanticChunker
from langchain_community.retrievers import WikipediaRetriever
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain.retrievers.weaviate_hybrid_search import WeaviateHybridSearchRetriever
from langchain.retrievers.merger_retriever import MergerRetriever
from langchain.document_transformers import (
    EmbeddingsClusteringFilter,
    EmbeddingsRedundantFilter,
)
from langchain.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain.prompts import ChatMessagePromptTemplate, PromptTemplate
from langchain.retrievers.document_compressors import DocumentCompressorPipeline
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import EmbeddingsFilter
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain.document_transformers import LongContextReorder
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain.llms import LlamaCpp
from langchain.retrievers.document_compressors import FlashrankRerank
from langchain_community.vectorstores import Chroma, FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from flashrank.Ranker import Ranker, RerankRequest
from dotenv import load_dotenv
from langchain.embeddings import HuggingFaceInferenceAPIEmbeddings
from sklearn.feature_extraction.text import TfidfVectorizer
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from transformers import ( AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline, )
from langchain.chains.query_constructor.base import (
    StructuredQueryOutputParser,
    get_query_constructor_prompt,
)
from langchain_core.prompts import ChatPromptTemplate
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CohereRerank
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain.embeddings import HuggingFaceEmbeddings, OpenAIEmbeddings,HuggingFaceBgeEmbeddings
from langchain.retrievers.document_compressors import LLMChainFilter
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CohereRerank
from langchain.prompts import PromptTemplate
from langchain import HuggingFacePipeline
from sklearn.metrics.pairwise import cosine_similarity
from langchain.chains import RetrievalQA
from google.colab import userdata
from google.colab import drive
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
from langchain.load import dumps, loads
import os
import re
from re import search
import textwrap
import torch
import weaviate
import pandas as pd
import numpy as np

load_dotenv()
OPENAI_API_KEY=userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
model = ChatOpenAI()

documents = TextLoader("/content/state_of_the_union.txt").load()
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=100)
texts = text_splitter.split_documents(documents)
texts

In [ ]:
def pretty_print_docs(docs):
  print(
    f"\n{'-' * 100}\n".join(
      [f"Document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]
    )
  )

retriever = FAISS.from_documents(texts, OpenAIEmbeddings().as_retriever())
docs = retriever.invoke("What did the president say about Ketanji Brown Jackson")

pretty_print_docs(docs)

In [ ]:
docs2 = retriever.invoke("What were the top three priorities outlined in the most recent State of the Union address?")
docs3 = retriever.invoke("How did the President propose to tackle the issue of climate change?")

pretty_print_docs(docs2)
pretty_print_docs(docs3)

In [ ]:
llm = OpenAI(temperature=0)
chain = RetrievalQA.from_chain_type(llm, retriever=retriever)
query="What were the top three priorities outlined in the most recent State of the Union address?"
chain.invoke(query)
print(chain.invoke(query)['result'])

In [ ]:
compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)
compressed_docs = compression_retriever.get_relevant_documents("What were the top three priorities outlined in the most recent State of the Union address?")
print(compressed_docs)
pretty_print_docs(compressed_docs)

In [ ]:
compressed_docs2 = compression_retriever.invoke("How did the President propose to tackle the issue of climate change?")
pretty_print_docs(compressed_docs2)

In [ ]:
filtered = LLMChainFilter.from_llm(model)
compression_retriever2 = ContextualCompressionRetriever(
    base_compressor=filtered, base_retriever=retriever
)
compressed_docs3 = compression_retriever2.invoke("What were the top three priorities outlined in the most recent State of the Union address?")
pretty_print_docs(compressed_docs3)

In [ ]:
original_contexts_len = len("\n\n".join([d.page_content for i, d in enumerate(docs2)]))
compressed_contexts_len = len("\n\n".join([d.page_content for i, d in enumerate(compressed_docs)]))

print("Original context length:", original_contexts_len)
print("Compressed context length:", compressed_contexts_len)
print("Compressed Ratio:", f"{original_contexts_len/(compressed_contexts_len + 1e-5):.2f}x")

In [ ]:
embeddings = OpenAIEmbeddings()
embeddings = EmbeddingsFilter(embeddings=embeddings)
compression_retriever3 = ContextualCompressionRetriever(
    base_compressor=filtered, base_retriever=retriever
)

compressed_contexts_len = len("\n\n".join([d.page_content for i, d in enumerate(compressed_docs)]))
compressed_docs4 = compression_retriever3.invoke("What were the top three priorities outlined in the most recent State of the Union address?")

pretty_print_docs(compressed_docs4)
print("Original context length:", original_contexts_len)
print("Compressed context length:", compressed_contexts_len)
print("Compressed Ratio:", f"{original_contexts_len/(compressed_contexts_len + 1e-5):.2f}x")

In [ ]:
splitter = CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=0,
    separators=". "
)
redundant_filter = EmbeddingsRedundantFilter(
    embeddings=embeddings,
)
relevant_filter = EmbeddingsFilter(
    embeddings=embeddings,
    similarity_threshold=0.76,
)
pipeline_compressor = DocumentCompressorPipeline(
    transformers=[splitter, redundant_filter, relevant_filter]
)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=pipeline_compressor, base_retriever=retriever
)

compressed_docs = compression_retriever.invoke("What were the top three priorities outlined in the most recent State of the Union address?")

pretty_print_docs(compressed_docs)

In [ ]:
chain = RetrievalQA.from_chain_type(llm=llm, retriever=compression_retriever)
query="What were the top three priorities outlined in the most recent State of the Union address?"

chain.invoke(query)
print(chain.invoke(query)['result'])

**Self-Querying Retriever**

In [ ]:
docs = [
    Document(
        page_content="A bunch of scientists bring back dinosaurs and mayhem breaks loose",
        metadata={"year": 1993, "rating": 7.7, "genre": "science fiction"},
    ),
    Document(
        page_content="Leo DiCaprio gets lost in a dream within a dream within a dream within a ...",
        metadata={"year": 2010, "director": "Christopher Nolan", "rating": 8.2},
    ),
    Document(
        page_content="A psychologist / detective gets lost in a series of dreams within dreams within dreams and Inception reused the idea",
        metadata={"year": 2006, "director": "Satoshi Kon", "rating": 8.6},
    ),
    Document(
        page_content="A bunch of normal-sized women are supremely wholesome and some men pine after them",
        metadata={"year": 2019, "director": "Greta Gerwig", "rating": 8.3},
    ),
    Document(
        page_content="Toys come alive and have a blast doing so",
        metadata={"year": 1995, "genre": "animated"},
    ),
    Document(
        page_content="A hacker discovers reality is a simulation and leads a rebellion against the machines controlling it.",
        metadata={"year": 1999, "director": "Lana Wachowski, Lilly Wachowski", "rating": 8.7, "genre": "science fiction"},
    ),
    Document(
        page_content="A young lion prince flees his kingdom only to learn the true meaning of responsibility and bravery.",
        metadata={"year": 1994, "rating": 8.5, "genre": "animated"},
    ),
    Document(
        page_content="Batman faces off against the Joker, a criminal mastermind who plunges Gotham into chaos.",
        metadata={"year": 2008, "director": "Christopher Nolan", "rating": 9.0, "genre": "action"},
    ),
    Document(
        page_content="A team of explorers travel through a wormhole in space in an attempt to ensure humanity's survival.",
        metadata={"year": 2014, "director": "Christopher Nolan", "rating": 8.6, "genre": "science fiction"},
    )
]

vectorstore = Chroma.from_documents(docs, embedding)
question1 = "Which 1994 animated movie has a rating of 8.5?"
question2 = "Which movie features Batman facing off against the Joker and who directed it?"
question3 = "What genre is the movie 'The Matrix' and who directed it?"


vectorstore.similarity_search(question1)
vectorstore.similarity_search(question2)

In [ ]:
def wrap_text(text, width=90):
  lines = text.split("\n")
  wrapped_lines = [textwrap.fill(line, width=width) for line in lines]
  wrapped_text = "\n".join(wrapped_lines)
  return wrapped_text

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)
chain=(
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

text_reply = chain.invoke(question1)
print(wrap_text(text_reply))

text_reply = chain.invoke("Tell me about the movie which have rating more than 7.")

text_reply = chain.invoke(question3)
print(wrap_text(text_reply))

In [ ]:
metadata_field_info = [
    AttributeInfo(
        name="genre",
        description="The genre of the movie. One of ['science fiction', 'comedy', 'drama', 'thriller', 'romance', 'action', 'animated']",
        type="string",
    ),
    AttributeInfo(
        name="year",
        description="The year the movie was released",
        type="integer",
    ),
    AttributeInfo(
        name="director",
        description="The name of the movie director",
        type="string",
    ),
    AttributeInfo(
        name="rating", description="A 1-10 rating for the movie", type="float"
    ),
]

document_content_description = "Brief summary of a movie"
prompt = get_query_constructor_prompt(
    document_content_description,
    metadata_field_info,
)

output_parser = StructuredQueryOutputParser.from_components()
query_constructor = prompt | model | output_parser
query_constructor.invoke(
    {
        "query": "What are some sci-fi movies from the 90's directed by Luc Besson about taxi drivers"
    }
)

print(prompt.format(query="dummy question"))

In [ ]:
retriever = SelfQueryRetriever(
    query_constructor=query_constructor,
    vectorstore=vectorstore,
    structured_query_translator=ChromaTranslator(),
)

retriever.invoke(
    "What's a movie after 1990 but before 2005 that's all about toys, and preferably is animated"
)

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

text_reply = chain.invoke("Tell me about the movie which have rating more than 7.")
print(wrap_text(text_reply))

### **Parent-Child Retriever**

In [ ]:
loaders = [
    TextLoader("/content/data/paul_graham_essay.txt"),
    TextLoader("/content/data/state_of_the_union.txt"),
]

docs = []
for loader in loaders:
  docs.extend(loader.load())

docs

In [ ]:
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400)
gemini_embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
vectorstore = Chroma(
    collection_name="full_documents",
    embedding_function=gemini_embeddings,
)

store = InMemoryStore()
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

retriever.add_documents(docs, ids=None)
list(store.yield_keys())
retrieved_docs= retriever.invoke("What did the president say about Ketanji Brown Jackson")

print(retrieved_docs[0].page_content)
print(len(retrieved_docs[0].page_content))

In [ ]:
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500)
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)

store1 = InMemoryStore()
retriever2 = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store1,
    child_splitter=child_splitter,
)

retriever2.add_documents(docs)
print(len(list(store1.yield_keys())))

retrieved_docs2= retriever2.invoke("What did the president say about Ketanji Brown Jackson")

print(retrieved_docs2)
len(retrieved_docs2[0].page_content)

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-1.5-pro")
qa = RetrievalQA.from_chain_type(llm=llm,
                                 chain_type="stuff",
                                 retriever=retriever2)

query = "What did the president say about Ketanji Brown Jackson"
qa.run(query)
result = llm.invoke("Write a ballad about LangChain")
print(result.content)